# exp079_public_artifact_replay_integrity_audit train

Public artifact replay integrity audit. This notebook inspects public notebook outputs and external inputs before any replay submission.

## Contents

1. Setup and configuration
2. Source and artifact integrity audit
3. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
from datetime import UTC, datetime

from public_artifact_integrity_audit import run_integrity_audit
from settings import EXPERIMENT_NAME, ExperimentPaths, load_config

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

audit_config = config.get("audit", {})
print("Experiment:", EXPERIMENT_NAME)
print("Route:", config["experiment"]["route"])
print("Mode:", audit_config.get("mode"))
print("Sample submission:", paths.sample_submission_path)
print("Artifacts:", paths.artifacts_dir)
print("Sources:", [source.get("name") for source in audit_config.get("source_specs", [])])

## 2. Source and artifact integrity audit

In [ ]:
summary = run_integrity_audit(
    config=config,
    root=paths.root,
    artifacts_dir=paths.artifacts_dir,
)

print("Audit status:", summary["status"])
print("Missing required sources:", len(summary["missing_required_sources"]))
print("Candidate submissions:", len(summary["submission_summaries"]))
print("Notebook inspections:", len(summary["notebook_inspections"]))
print("Pairwise distances:", len(summary["pairwise_distances"]))

## 3. Metrics and artifacts

In [ ]:
metrics = {
    "experiment": EXPERIMENT_NAME,
    "status": summary["status"],
    "cv": None,
    "public_lb": None,
    "private_lb": None,
    "metric": "integrity_audit",
    "updated_at": datetime.now(UTC).isoformat(),
    "notes": {
        "candidate_submission_count": len(summary["submission_summaries"]),
        "notebook_inspection_count": len(summary["notebook_inspections"]),
        "missing_required_source_count": len(summary["missing_required_sources"]),
        "pairwise_distance_count": len(summary["pairwise_distances"]),
        "direct_submit_allowed": False,
    },
}
paths.metrics_path.write_text(json.dumps(metrics, indent=2, sort_keys=True) + "\n")

print(json.dumps(metrics, indent=2, sort_keys=True))